In [1]:
import sys
import os
sys.path.append('/project/GCRB/Hon_lab/s223695/Data_project/Perturb_seq_shared/')

import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.cluster import KMeans
import scipy.stats
from scipy.stats import hypergeom
from sklearn.metrics import pairwise_distances
from itertools import combinations


import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams.update({'axes.labelsize' : 'large',
                     'pdf.fonttype':42
                    }) 
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm
from tqdm.contrib.concurrent import process_map

import multiprocessing as mp

import gc
import warnings
import time
import pickle
import json

from sklearn.metrics import pairwise_distances
import torch

from importlib import reload
import util_functions

/project/GCRB/Hon_lab/s223695/anaconda3/envs/scanpy_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sc.__version__

'1.9.8'

In [3]:
!nvidia-smi

Sat Oct  5 11:21:53 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           Off |   00000000:3B:00.0 Off |                    0 |
| N/A   27C    P0             24W /  250W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!free -m

              total        used        free      shared  buff/cache   available
Mem:         385502        9090      374622         324        1789      374903
Swap:        131071        1934      129137


<h3>Load PCA matrix and gRNA-Cell name dictionary</h3>

In [5]:
json_fp = "./config.json"
with open(json_fp, 'r') as fp:
    config = json.load(fp)
    
input_file = config["input_data"]["input_file"]
sgRNA_file = config["input_data"]["sgRNA_file"]

gRNA_dict_file = config["user_defined_data"]["gRNA_dict_file"]
pca_file = config["user_defined_data"]["pca_file"]
annotation_file = config["user_defined_data"]["annotation_file"]

# parameter for gRNA filtering 
combi_count= config["gRNA_filtering"]["combi_count"]

# parameter for gRNA filtering 
batch_num_basic = config["gRNA_filtering"]["batch_num_basic"]
total_permute_disco = config["gRNA_filtering"]["total_permute_disco"]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

annotation_df = pd.read_csv(annotation_file)

(X,gRNA_dict) = util_functions.load_files(input_file,sgRNA_file,gRNA_dict_file,pca_file)

read input
read pickle
read from dictionary


In [6]:
gRNA_region_dict = {}
count_region_dict = {}

for index,row in annotation_df.iterrows():
    if row.protospacer_target in gRNA_dict.keys():
        if row.intended_target_region in gRNA_region_dict.keys():
            gRNA_region_dict[row.intended_target_region] += [row.protospacer_target]
        else:
            gRNA_region_dict[row.intended_target_region] = [row.protospacer_target]
            
for key in gRNA_region_dict.keys():
    gRNA_region_dict[key] = np.unique(gRNA_region_dict[key])
    count_region_dict[key] = len(gRNA_region_dict[key])

In [7]:
np.unique(annotation_df["source"])

array(['CHDgene.au', 'OpenTargets', 'OpenTargets,CHDgene.au', 'TF',
       'TFPerturbSeq', 'TimeDep', 'TimeDep,OpenTargets',
       'TimeDep,OpenTargets,CHDgene.au', 'TimeDep,TF',
       'TimeDep,TFPerturbSeq'], dtype=object)

<h3>Calculate e-distance between gRNA per target</h3>

In [8]:
result_df_dict = {}

In [ ]:
#compute every pair in sgRNA make rank-distance plot
test_region_np = np.array(list(gRNA_region_dict.keys()))
pbar = tqdm(test_region_np)

for target in pbar:
    total_combis_tmp = []
    
    gRNA_list_target = np.array(gRNA_region_dict[target])
    total_gRNA_num = len(gRNA_list_target)
    
    if total_gRNA_num>6:
        for combis_tmp in combinations(gRNA_list_target,combi_count):
            combis_tmp = np.array(combis_tmp)
            for first_group_num in range(1,combi_count):
                first_group_list = combinations(combis_tmp,first_group_num)
                for first_group in first_group_list:
                    second_group = combis_tmp[~np.isin(combis_tmp,first_group)]
                    total_combis_tmp += [(list(first_group),list(second_group))]
    else:
        for num_count in range(1,total_gRNA_num):
            for combis_tmp in combinations(gRNA_list_target,num_count):
                other_gRNA = gRNA_list_target[~np.isin(gRNA_list_target,combis_tmp)]
                for num_other_count in range(1,total_gRNA_num):
                    for other_tmp in combinations(other_gRNA,num_other_count):
                        total_combis_tmp += [(list(combis_tmp),list(other_tmp))]

    total_combis = util_functions.get_unique_list(total_combis_tmp)
    res = []
    for index_combi,(combi_test1,combi_test2) in enumerate(total_combis):
        if index_combi % 100 == 0:
            pbar.set_postfix({"processing combi":index_combi+1,
                              "total combi":len(total_combis),
                              "mode": "GPU"
                         })
        cell_test1 = np.unique(np.concatenate([gRNA_dict[name] for name in combi_test1]))
        cell_test2 = np.unique(np.concatenate([gRNA_dict[name] for name in combi_test2]))
        total_cell_num = len(cell_test1)+len(cell_test2)
        
        batch_num = 1
        try:
            obs_edist = util_functions.permutation_test(X,cell_test1,cell_test2,device,
                                                        1,1,return_permute=False).cpu()
        except:
            pbar.set_postfix({"processing combi":index_combi+1,
                              "total combi":len(total_combis),
                              "mode": "out of memory, CPU"
                             })
            try:
                obs_edist = util_functions.permutation_test(X,cell_test1,cell_test2,"cpu",
                                                            1,1,return_permute=False).cpu()
            except:
                print("data too large",target,index_combi)
                res += [-1]
                continue
                
        res += [obs_edist.item()]

    result_df = pd.DataFrame(zip(res,total_combis),columns=["e_dist","combis"])
    result_df = result_df.sort_values(["e_dist"]).reset_index(drop=True)
    result_df["rank"] =result_df.index.tolist()

    for gRNA_name_tmp in gRNA_list_target:
        result_df[gRNA_name_tmp] = result_df["combis"].apply(lambda x: (gRNA_name_tmp in x[0]) or (gRNA_name_tmp in x[1]))
    result_df_dict[target] = result_df

 14%|█▍        | 216/1546 [01:42<11:09,  1.99it/s, processing combi=101, total combi=301, mode=GPU]

In [ ]:
print(test_region_np.shape)
print(len(result_df_dict))

In [ ]:
target_arr = np.array(list(result_df_dict.keys()))

In [ ]:
p_val_dict = {}

test_three_gRNA = False
disco_val_dict = {}

for target in tqdm(result_df_dict.keys()):
    result_df = result_df_dict[target]
    total_gRNA_list = gRNA_region_dict[target]
    total_cell_list = [gRNA_dict[i] for i in total_gRNA_list]
    total_cell_num = len(np.concatenate(total_cell_list))
    
    if total_cell_num > 20000:
        batch_num = 1
    if total_cell_num > 5000:
        batch_num = batch_num_basic //4
    elif  total_cell_num > 300:
         batch_num = batch_num_basic //2
    else :
        batch_num = batch_num_basic
    
    try:        
        obs_fvalue, fvalue_list = util_functions.disco_test(X,total_cell_list,device,batch_num=batch_num)
        disco_pvalue = (fvalue_list>obs_fvalue).sum().item()/total_permute_disco
    except:
        print("out of memory, use cpu:",target)
        try:
            obs_fvalue, fvalue_list = util_functions.disco_test(X,total_cell_list,"cpu",batch_num=batch_num)
            disco_pvalue = (fvalue_list>obs_fvalue).sum().item()/total_permute_disco
        except:
            print("data is too large, skip disco test for",target)
            disco_pvalue = 0
    disco_val_dict[target] = disco_pvalue
    
    if disco_pvalue > 0.05 or len(total_gRNA_list)==2:
        for gRNA_name_tmp in total_gRNA_list:
            p_val_dict[gRNA_name_tmp] = 1.0
        continue
    else:
        num_sig_diff = result_df.shape[0] // 10
        result_df_right = result_df.iloc[(result_df.shape[0]-num_sig_diff):,:]

        for gRNA_name_tmp in gRNA_region_dict[target]:
            [M, n, N, x] = [result_df.shape[0],np.sum(result_df[gRNA_name_tmp]),num_sig_diff,np.sum(result_df_right[gRNA_name_tmp])]
            
            prb = np.round(1-hypergeom.cdf(x, M, n, N),4)
            p_val_dict[gRNA_name_tmp] = prb

outlier_df = pd.Series(p_val_dict)
outlier_df.name = "pval_outlier"

outlier_df.to_csv("Step1_sgRNA_outlier_pval.csv")

In [ ]:
cols = 2

with PdfPages("Step1_gRNA_outlier_pval.pdf") as pdf:
    for target in tqdm(result_df_dict.keys()):
        gRNA_num = len(gRNA_region_dict[target])
        if gRNA_num == 1:
            continue
        
        rows = gRNA_num//cols + 1
        fig, ax = plt.subplots(rows,cols, figsize=(16,4*rows))
        
        result_df = result_df_dict[target]
        
        for index, gRNA_name_tmp in enumerate(gRNA_region_dict[target]):
            col_index = index % cols
            row_index = index // cols
            sns.scatterplot(data=result_df,x="rank",y="e_dist",hue=gRNA_name_tmp,ax=ax[row_index,col_index])
            ax[row_index,col_index].text(0,(result_df["e_dist"].max()+result_df["e_dist"].min())/2,
                                         outlier_df[gRNA_name_tmp])
        pdf.savefig()
        plt.close()

<h3>Filtering out bad gRNA using Bonferroni correction</h3>

In [25]:
outlier_df_clear = outlier_df[outlier_df>0.05]
outlier_df_clear = pd.DataFrame(outlier_df_clear)


In [26]:
outlier_df_clear["target_region"] = [annotation_df[annotation_df["protospacer_target"]==key]["intended_target_region"].values[0]
                                         for key in outlier_df_clear.index.to_list()]
outlier_df_clear["target_name"] = [annotation_df[annotation_df["protospacer_target"]==key]["gene_target"].values[0]
                                         for key in outlier_df_clear.index.to_list()]

In [28]:
outlier_df_clear.to_csv("Step1_sgRNA_outlier_pval_clear.csv")